# 05 - Vérification des raccourcis (biais) dans les données

**Projet :** Détection de prompt injection et jailbreak — **Membre 1**

## Pourquoi ce notebook ?
Un modèle peut avoir un excellent score **sans comprendre ce qu'est une attaque**, s'il trouve un *raccourci* dans les données.
Exemple : si tous les prompts malveillants sont longs, « texte long = attaque » suffit.

On teste trois raccourcis possibles avec des modèles volontairement **naïfs** :
1. **Longueur** du texte seulement.
2. **Langue** seulement (« pas anglais = attaque »).
3. **Source** du dataset seulement.

Si un modèle naïf atteint presque le score de la baseline, alors le score de la baseline ne prouve pas grand-chose.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

while not os.path.exists("data/processed") and os.getcwd() != "/":
    os.chdir("..")
os.makedirs("results/figures", exist_ok=True)

P = "data/processed"
train_seen  = pd.read_csv(f"{P}/train_seen.csv")
test_seen   = pd.read_csv(f"{P}/test_seen.csv")
test_unseen = pd.read_csv(f"{P}/test_unseen_family.csv")
SEED = 42

def evaluate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y_true, y_pred),
            "detection_rate": tp / (tp + fn) if (tp + fn) else float("nan"),
            "fpr": fp / (fp + tn) if (fp + tn) else float("nan"),
            "f1": f1_score(y_true, y_pred, zero_division=0)}

## 1. Les données en chiffres : longueur, langue, source

In [ ]:
for d in (train_seen, test_seen, test_unseen):
    d["n_chars"] = d["text"].astype(str).str.len()
    d["n_words"] = d["text"].astype(str).str.split().str.len()

print("Longueur médiane (caractères) par classe, dans train_seen :")
print(train_seen.groupby("label_binary")["n_chars"].median().rename({0: "légitime", 1: "malveillant"}).to_string())

print("\nLangue x classe (train_seen) :")
display(pd.crosstab(train_seen.language, train_seen.label_binary).rename(columns={0: "légitime", 1: "malveillant"}))

print("Source x classe (train_seen) :")
display(pd.crosstab(train_seen.source, train_seen.label_binary).rename(columns={0: "légitime", 1: "malveillant"}))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for lab, name in [(0, "légitime"), (1, "malveillant")]:
    ax.hist(np.log10(train_seen[train_seen.label_binary == lab]["n_chars"] + 1), bins=50, alpha=0.6, label=name)
ax.set_xlabel("log10(nombre de caractères)")
ax.set_ylabel("nombre de prompts")
ax.set_title("Distribution des longueurs (train_seen)")
ax.legend()
plt.tight_layout()
plt.savefig("results/figures/length_distribution.png", dpi=150)
plt.show()

## 2. Modèles naïfs
Chacun est entraîné sur `train_seen` avec **une seule information** (longueur, langue ou source).

In [ ]:
rows = []

# (1) Longueur seulement
def length_feats(d):
    return np.c_[np.log1p(d["n_chars"]), np.log1p(d["n_words"])]
sc = StandardScaler().fit(length_feats(train_seen))
m_len = LogisticRegression(class_weight="balanced", random_state=SEED).fit(sc.transform(length_feats(train_seen)), train_seen.label_binary)
for name, d in [("test_seen", test_seen), ("test_unseen_family", test_unseen)]:
    rows.append({"modele_naif": "longueur seule", "ensemble": name,
                 **evaluate(d.label_binary, m_len.predict(sc.transform(length_feats(d))))})

# (2) Langue seulement : règle « pas anglais = malveillant »
for name, d in [("test_seen", test_seen), ("test_unseen_family", test_unseen)]:
    rows.append({"modele_naif": "regle: langue != en", "ensemble": name,
                 **evaluate(d.label_binary, (d.language != "en").astype(int))})

# (3) Source seulement
oh = OneHotEncoder(handle_unknown="ignore").fit(train_seen[["source"]])
m_src = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED).fit(oh.transform(train_seen[["source"]]), train_seen.label_binary)
for name, d in [("test_seen", test_seen), ("test_unseen_family", test_unseen)]:
    rows.append({"modele_naif": "source seule", "ensemble": name,
                 **evaluate(d.label_binary, m_src.predict(oh.transform(d[["source"]])))})

shortcuts = pd.DataFrame(rows)
display(shortcuts.round(4))

## 3. Comparaison avec la baseline TF-IDF

In [ ]:
base = pd.read_csv("results/baseline_metrics.csv", index_col=0)
base_rows = []
for name in ["test_seen", "test_unseen_family"]:
    r = base.loc[name]
    base_rows.append({"modele_naif": "BASELINE TF-IDF + LogReg", "ensemble": name,
                      "accuracy": r["accuracy"], "detection_rate": r["detection_rate"], "fpr": r["fpr"], "f1": r["f1"]})
comparison = pd.concat([shortcuts, pd.DataFrame(base_rows)], ignore_index=True)
display(comparison.round(4))
comparison.to_csv("results/bias_checks.csv", index=False)

In [ ]:
pivot = comparison.pivot(index="modele_naif", columns="ensemble", values="f1")
ax = pivot.plot(kind="barh", figsize=(9, 4))
ax.set_xlabel("F1")
ax.set_title("F1 : modèles naïfs vs baseline TF-IDF")
plt.tight_layout()
plt.savefig("results/figures/bias_checks_f1.png", dpi=150)
plt.show()

## 4. Comment interpréter

**Sur `test_seen` (attaques connues)**
- Les modèles naïfs obtiennent des scores non négligeables (F1 ≈ 0,78 avec la seule longueur, ≈ 0,82 avec la seule source), ce qui prouve que **les raccourcis existent** dans ce dataset.
- La baseline TF-IDF fait nettement mieux (F1 ≈ 0,97) : elle apprend donc aussi autre chose que ces raccourcis. Mais une partie de son score reste à prendre avec prudence.

**Sur `test_unseen_family` (jailbreaks jamais vus)**
- Les modèles « longueur » et « source » ont un **taux de détection très élevé** (≈ 97 % et ≈ 98 %), mais au prix d'un **taux de faux positifs de 15 % et 11 %**, contre ≈ 1,3 % pour la baseline.
- Explication : les jailbreaks sont des textes **très longs** et viennent d'une **source jamais vue**. « Texte long » ou « source inconnue » suffit à les repérer, sans comprendre l'attaque. C'est exactement un raccourci.
- Conclusion : le **taux de détection seul est trompeur**. Il faut toujours le lire avec le taux de faux positifs. La baseline TF-IDF obtient le meilleur F1 sur la famille non vue (≈ 0,90 contre ≈ 0,70 et ≈ 0,77).

**Limites connues (à écrire dans le rapport)**
1. Tous les prompts non anglais du dataset sont malveillants et il n'y a aucun prompt légitime non anglais. On ne peut donc pas mesurer les faux positifs sur des prompts légitimes en français. Les coefficients de la baseline (notebook 04) contiennent d'ailleurs des fragments comme `' un'`, `' de '`, `' yo'`, `' di'` : signe que la langue sert déjà d'indice.
2. Presque tous les jailbreaks viennent d'une seule source, donc « famille retirée » et « source retirée » sont confondues.
3. Une amélioration possible, à décider en équipe : ajouter des exemples légitimes en français (voir `data/README.md`).